In [9]:
import recordlinkage
import pandas as pd
import re

social_dataset = pd.read_excel('../../Mediated Schema Excels/socialmedia_schema.xlsx')

social_dataset = social_dataset[
    social_dataset['Other'].notnull() & (social_dataset['Other'].str.strip() != '')
]

def tokenize_and_clean_url(url):
    if type(url) is not str:
        url = str(url)
    url = url.lower().replace("http://", "").replace("https://", "").rstrip("/")
    tokens = re.split(r'\W+', url)
    tokens = sorted(token for token in tokens if token)
    return " ".join(tokens)

social_dataset['Other_clean'] = social_dataset['Other'].apply(tokenize_and_clean_url)

def social_blocking():
    indexer = recordlinkage.Index()
    indexer.sortedneighbourhood('Name', window=3)
    candidate_links = indexer.index(social_dataset)
    compare = recordlinkage.Compare()
    compare.string('Other_clean', 'Other_clean', method='jarowinkler', label='url_similarity')
    compare_vectors = compare.compute(candidate_links, social_dataset)
    
    matched_pairs = compare_vectors[compare_vectors['url_similarity'] > 0.91]

    
    df = pd.DataFrame({
        "row_index_1": social_dataset.index.get_indexer(matched_pairs.index.get_level_values(0)) + 2,
        "row_index_2": social_dataset.index.get_indexer(matched_pairs.index.get_level_values(1)) + 2,
        "name_company_1": social_dataset.loc[matched_pairs.index.get_level_values(0), "Name"].values,
        "name_company_2": social_dataset.loc[matched_pairs.index.get_level_values(1), "Name"].values,
        "other_1": social_dataset.loc[matched_pairs.index.get_level_values(0), "Other"].values,
        "other_2": social_dataset.loc[matched_pairs.index.get_level_values(1), "Other"].values,
        "similarity_score": matched_pairs["url_similarity"].values,
        "is_match": 1
    })

    df.to_excel("../../blocking_excels/socialmedia_blocking.xlsx", index=False)

In [10]:
social_blocking()